# Ethiopia EDA

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats



# Load Data

In [2]:
df = pd.read_csv("../data/kenya.csv")
df["Country"] = "Kenya"
df.head()


,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country
0,2015,1,19.56,28.99,12.09,16.90,0.00,45.32,3.12,4.76,83.68,6.88,Kenya
1,2015,2,19.63,29.77,11.04,18.73,0.00,38.76,3.23,4.35,83.67,5.85,Kenya
2,2015,3,20.40,30.57,11.71,18.86,0.00,41.75,3.46,4.68,83.69,6.65,Kenya
3,2015,4,21.33,31.20,13.02,18.18,3.49,51.87,2.29,4.00,83.62,8.60,Kenya
4,2015,5,20.41,29.52,12.38,17.14,1.79,48.04,1.77,4.05,83.54,7.64,Kenya


In [3]:
df["Date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")
df["Month"] = df["Date"].dt.month
df.head()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country,Date,Month
0,2015,1,19.56,28.99,12.09,16.90,0.00,45.32,3.12,4.76,83.68,6.88,Kenya,2015-01-01,1
1,2015,2,19.63,29.77,11.04,18.73,0.00,38.76,3.23,4.35,83.67,5.85,Kenya,2015-01-02,1
2,2015,3,20.40,30.57,11.71,18.86,0.00,41.75,3.46,4.68,83.69,6.65,Kenya,2015-01-03,1
3,2015,4,21.33,31.20,13.02,18.18,3.49,51.87,2.29,4.00,83.62,8.60,Kenya,2015-01-04,1
4,2015,5,20.41,29.52,12.38,17.14,1.79,48.04,1.77,4.05,83.54,7.64,Kenya,2015-01-05,1


## Clean

In [4]:
df.replace(-999, np.nan, inplace=True)

In [5]:
print(df.duplicated(), '\n')
print(f"How many duplicates: {df.duplicated().sum()}")

df = df.drop_duplicates()

0       False
1       False
2       False
3       False
4       False
        ...  
4103    False
4104    False
4105    False
4106    False
4107    False
Length: 4108, dtype: bool 

How many duplicates: 0


In [6]:
df.describe()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Date,Month
count,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108,4108.000000
mean,2020.131451,180.121227,20.427600,27.838717,14.673169,13.165548,1.468162,65.845355,3.061765,4.375241,83.724335,11.052539,2020-08-15 12:00:00,6.423564
min,2015.000000,1.000000,15.260000,18.880000,8.970000,4.110000,0.000000,28.420000,0.610000,1.160000,83.310000,4.780000,2015-01-01 00:00:00,1.000000
25%,2017.000000,86.000000,19.460000,26.297500,13.700000,11.467500,0.100000,58.677500,2.420000,3.670000,83.630000,9.880000,2017-10-23 18:00:00,3.000000
50%,2020.000000,179.000000,20.360000,27.875000,14.750000,13.260000,0.380000,66.220000,3.140000,4.430000,83.720000,11.005000,2020-08-15 12:00:00,6.000000
75%,2023.000000,272.000000,21.400000,29.520000,15.750000,15.050000,1.360000,73.280000,3.720000,5.090000,83.810000,12.350000,2023-06-08 06:00:00,9.000000
max,2026.000000,366.000000,25.400000,34.270000,18.750000,20.120000,51.650000,91.070000,5.280000,7.590000,84.170000,15.040000,2026-03-31 00:00:00,12.000000
std,3.248907,106.294767,1.440824,2.358770,1.415691,2.605174,3.180228,9.934196,0.853218,0.992156,0.126391,1.607151,NaN,3.477046


# Summary statistics
this is the summmary statistics of all numeric columns

In [7]:
missing = df.isna().sum()#this will give us the percentage of missing values in each column
missing_percent = (missing/len(df)) * 100

high_missing = missing_percent[missing_percent > 5]

if high_missing.empty:
    summary = pd.DataFrame(
        {
            "Missing Count": missing,
            "Missing Percent": missing_percent
        }
    )
    display(summary)

,Missing Count,Missing Percent
YEAR,0,0.0
DOY,0,0.0
T2M,0,0.0
T2M_MAX,0,0.0
T2M_MIN,0,0.0
T2M_RANGE,0,0.0
PRECTOTCORR,0,0.0
RH2M,0,0.0
WS2M,0,0.0
WS2M_MAX,0,0.0


## Outliers

In [8]:
cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

z_scores = np.abs(stats.zscore(df[cols].dropna(),nan_policy='omit'))
z_df = pd.DataFrame(z_scores, index=df[cols].dropna().index, columns=cols)

outlier_mask = (z_scores > 3).any(axis=1)
print((z_scores > 3).sum())
'''

df = df[df.isna().mean(axis=1) <= 0.3].copy()
df[weather_cols] = df[weather_cols].ffill().bfill()

if 'T2M_RANGE' not in df.columns:
        df['T2M_RANGE'] = df['T2M_MAX'] - df['T2M_MIN']
        
df.to_csv('../data/ethiopia_clean.csv', index=False)
print('saved')



'''

124


"\n\ndf = df[df.isna().mean(axis=1) <= 0.3].copy()\ndf[weather_cols] = df[weather_cols].ffill().bfill()\n\nif 'T2M_RANGE' not in df.columns:\n        df['T2M_RANGE'] = df['T2M_MAX'] - df['T2M_MIN']\n\ndf.to_csv('../data/ethiopia_clean.csv', index=False)\nprint('saved')\n\n\n\n"

### Outlier Treatment: Retain

**Decision:** I have decided to **retain** the 124 identified outlier rows.

**Reasoning:**
* **Data Authenticity:** In weather data (`T2M`, `PRECTOTCORR`), extremes are often real environmental events (storms or heatwaves) rather than errors.
* **Seasonal Integrity:** Removing these would bias the seasonal analysis by hiding the natural variability of the climate.
* **Model Robustness:** Retaining them ensures the analysis reflects real-world conditions, including high-impact weather peaks.

In [9]:
for col in cols:
    upper = df[col].mean() + 3 * df[col].std()
    lower = df[col].mean() - 3 * df[col].std()
    df[col] = np.clip(df[col], lower, upper)

# 1. Drop rows with more than 30% missing values
df = df.dropna(thresh=int(0.7 * len(df.columns)))
# 2. Forward-fill remaining weather variables
df.fillna(method="ffill", inplace=True)
# 3. Export to CSV (replace 'Ethiopia' with your country name)
df.to_csv("../data/ethiopia_clean.csv", index=False)

C:\Users\HP\AppData\Local\Temp\ipykernel_15008\3545118969.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", inplace=True)


## Time series

1. Monthly Average T2M (Line Chart)


In [ ]:
monthly_t2m = df.groupby(["YEAR", "Month"])["T2M"].mean().reset_index()

#Create a proper date for plotting
monthly_t2m["Date"] = pd.to_datetime(monthly_t2m["YEAR"].astype(str) + "-" + monthly_t2m["Month"].astype(str))

#Plot line chart
plt.figure(figsize=(12,6))
plt.plot(monthly_t2m["Date"], monthly_t2m["T2M"])
plt.title("Monthly Average Temperature (T2M) (2015–2026)")
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.show()



Annotate Warmest & Coolest Months

In [ ]:
#Annotate Warmest & Coolest Months
max_row = monthly_t2m.loc[monthly_t2m["T2M"].idxmax()]
min_row = monthly_t2m.loc[monthly_t2m["T2M"].idxmin()]

plt.figure(figsize=(12,6))
plt.plot(monthly_t2m["Date"], monthly_t2m["T2M"])

plt.scatter(max_row["Date"], max_row["T2M"])
plt.text(max_row["Date"], max_row["T2M"], "Warmest", color="red")

plt.scatter(min_row["Date"], min_row["T2M"])
plt.text(min_row["Date"], min_row["T2M"], "Coolest", color="blue")

plt.title("Monthly Avg Temperature with Annotations")
plt.show()

In [ ]:

#Monthly Total PRECTOTCORR (Bar Chart)
monthly_precip = df.groupby('Month')['PRECTOTCORR'].sum()
plt.figure()
plt.bar(monthly_precip.index, monthly_precip.values, color='skyblue')
plt.title('Monthly total PRECTOTCORR')
plt.xlabel('Month')
plt.ylabel('PRECTOTCORR')
plt.show()



## Correlation

Correlation Heatmap

In [ ]:
num = df.select_dtypes(include=[np.number])
corr = num.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation heatmap')
plt.show()

Scatter Plot: T2M vs RH2M

In [ ]:
print(corr.unstack().sort_values(ascending=False)[len(corr):len(corr)+3])
plt.figure()
sns.scatterplot(data=df, x='T2M', y='RH2M', alpha=0.5)
plt.title('T2M vs RH2M')
plt.show()

Scatter Plot: T2M_RANGE vs WS2M

In [ ]:

plt.figure()
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', alpha=0.5)
plt.title('T2M_RANGE vs WS2M')
plt.show()


## Distribution

Histogram of PRECTOTCORR

In [ ]:
sns.histplot(df["PRECTOTCORR"], bins=50)
plt.title("Precipitation Distribution")
plt.show()

If skewed → log scale
Right-skewed (positively skewed)

In [ ]:
sns.histplot(np.log1p(df["PRECTOTCORR"]), bins=50)
plt.title("Log-Scaled Precipitation")
plt.show()

Bubble Chart

In [ ]:
plt.figure()
df2 = df[df['PRECTOTCORR'] > 0]
sns.scatterplot(data=df2, x='T2M', y='RH2M', size='PRECTOTCORR', sizes=(20, 300), alpha=0.5, hue='PRECTOTCORR', palette='viridis', legend=False)
plt.title('T2M vs RH2M bubble')
plt.show()